In [197]:
import pandas as pd
import numpy as np

### Development Indicators

Development indicators were obtained from three data sources—UNDP, the World Development Indicators (WDI), and Our World in Data. These sources differ in metadata structure, variable naming conventions, and data types. Each dataset was first cleaned and transformed into an analogous structure. The harmonized datasets were subsequently merged using ISO alpha-3 country codes and year identifiers.  

#### From our world in data

For datasets from our world in data datasets, the following steps are applied within a for loop:
1. Years outside the range of 1990 to 2020 are pruned out. 
2. The column name 'Code' is changed to 'iso_o', and 'Entity' is changed to 'country'

In [198]:
literacy = pd.read_csv('../data/raw/cross-country-literacy-rates.csv')
gini = pd.read_csv('../data/raw/economic-inequality-gini-index.csv')
institute = pd.read_csv('../data/raw/government-effectiveness-vs-electoral-democracy-index.csv')
corruption = pd.read_csv('../data/raw/political-corruption-index.csv')

In [199]:
owd = {'literacy': literacy,
        'gini': gini,
        'institute': institute,
        'corruption': corruption}

In [200]:
for series, data in owd.items():
    print(series, data['Year'].min(), data['Year'].max())

literacy 1475 2023
gini 1963 2024
institute 1789 2024
corruption 1789 2024


In [201]:
for series, data in owd.items():
    data = data[(data['Year'] >= 1990) & (data['Year'] <= 2020)]
    data = data.rename(columns={'Code': 'iso_o', 'Entity': 'country', 'Year':'year'})
    globals()[series] = data

In [202]:
literacy.head(2)

,country,iso_o,year,Literacy rate
2,Afghanistan,AFG,2011,31.00000
3,Afghanistan,AFG,2015,33.75384


In [203]:
print(literacy['iso_o'].nunique())
print(gini['iso_o'].nunique())
print(institute['iso_o'].nunique())
print(corruption['iso_o'].nunique())

159
168
181
181


In [204]:
merged_owd = literacy.merge(gini, on = ['country','iso_o','year'], how = 'left').\
            merge(institute, on = ['country','iso_o','year'], how = 'left').\
            merge(corruption, on = ['country','iso_o','year'], how = 'left')

In [205]:
merged_owd = merged_owd.rename(columns = {'Literacy rate': 'literacy_rate', 'World region according to OWID_x':'region', 
                                          'Electoral Democracy ndex':'electoral_dem_index', 'Government Effectiveness Index':'government_effec_index',
                                          'Political Corruption Index': 'political_corruption_index', 'Gini coefficient':'gini'})

In [206]:
merged_owd = merged_owd.drop(columns= ['World region according to OWID_y', 'World region according to OWID'])

In [207]:
merged_owd.head(2)

,country,iso_o,year,literacy_rate,gini,region,electoral_dem_index,government_effec_index,political_corruption_index
0,Afghanistan,AFG,2011,31.00000,NaN,NaN,0.358,-1.474,0.958
1,Afghanistan,AFG,2015,33.75384,NaN,NaN,0.353,-1.396,0.904


In [208]:
regional_entries = ['Central and Southern Asia (SDG)', 'East Asia and Pacific (WB)','Eastern and South-Eastern Asia (SDG)',
 'Europe and Central Asia (WB)','Europe and Northern America (SDG)','Latin America and Caribbean (WB)','Latin America and the Caribbean (SDG)',
 'Low-income countries','Lower-middle-income countries', 'Middle East and North Africa (WB)', 'Middle-income countries','North America (WB)',
 'North Korea','Northern Africa and Western Asia (SDG)','Oceania (excluding Australia and New Zealand) (SDG)','Saint Helena','South Asia (WB)',
 'Sub-Saharan Africa (SDG)','Sub-Saharan Africa (WB)', 'Upper-middle-income countries', 'World']

In [209]:
merged_owd = merged_owd[~merged_owd['country'].isin(regional_entries)]

In [210]:
merged_owd.isna().sum()

country                         0
iso_o                           0
year                            0
literacy_rate                   0
gini                          459
region                        459
electoral_dem_index            53
government_effec_index        152
political_corruption_index     53
dtype: int64

In [211]:
print(f"Earliest year is {merged_owd['year'].min()} and the latest year is {merged_owd['year'].max()}.")

Earliest year is 1990 and the latest year is 2020.


#### From WDI

For WDI datasets, the following steps are applied within a for loop:
1. The column name 'Country Code' is changed to 'iso_o'
2. All datasets are pruned from the row 266. 
3. Year column names are cleaned. 
4. Year column values are converted to numeric and rounded upto 3 decimal places. '..' is replaced by NaN. 
5. Meta data columns are dropped. 
6. Dataframes are stacked to long format.
7. Years other than the range from 1990 to 2020 are trimmed out.

In [212]:
GDP_pc = pd.read_csv('../data/raw/11606a24-87be-4d81-92b2-a318c1453d9d_Series - Metadata.csv')
employment = pd.read_csv('../data/raw/4fd3be7e-f4f1-48fd-85ed-a24eaaf4dc17_Series - Metadata.csv', encoding='ISO-8859-1')
lfp = pd.read_csv('../data/raw/519680ac-db02-461a-9537-48d38bdc8303_Series - Metadata.csv', encoding='ISO-8859-1')
hci = pd.read_csv('../data/raw/a74e5e00-ba83-43b1-9754-f485801a9114_Series - Metadata.csv', encoding='ISO-8859-1')
mortality = pd.read_csv('../data/raw/1b88092e-3767-472c-943e-c094fe6c827b_Series - Metadata.csv')
primary_enroll = pd.read_csv('../data/raw/b6d35734-229f-4801-b42a-ef191a199de3_Series - Metadata.csv')
secondary_enroll = pd.read_csv('../data/raw/627ff230-d9e1-4120-9289-bb57fd3014aa_Series - Metadata.csv')
tertiary_enroll = pd.read_csv('../data/raw/22a7c73c-cd6f-44f9-bd0c-48b02c936c21_Series - Metadata.csv')
maternal_death_risk = pd.read_csv('../data/raw/2ca0cac3-0083-4c1e-95b1-9d0b550db279_Series - Metadata.csv')
mmr = pd.read_csv('../data/raw/9497afc7-e502-462f-ad75-adedc52d7e50_Series - Metadata.csv')
undernourishment = pd.read_csv('../data/raw/0c1e1631-f1d4-4aa0-9a5c-908a09804a04_Series - Metadata.csv')
DPT_immunization = pd.read_csv("../data/raw/143d1677-a021-4d3e-8712-e80da8282f20_Series - Metadata.csv")
population_growth = pd.read_csv("../data/raw/06d79aec-e87a-4af8-96f9-1f2b34d01ee9_Series - Metadata.csv")

In [213]:
undernourishment.head(2)

,Series Name,Series Code,Country Name,Country Code,1990 [YR1990],1991 [YR1991],1992 [YR1992],1993 [YR1993],1994 [YR1994],1995 [YR1995],...,2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024]
0,Prevalence of undernourishment (% of population),SN.ITK.DEFC.ZS,Afghanistan,AFG,..,..,..,..,..,..,...,19.9,20.4,21.3,22.6,25.1,27.5,28.8,28.9,28.1,..
1,Prevalence of undernourishment (% of population),SN.ITK.DEFC.ZS,Albania,ALB,..,..,..,..,..,..,...,4.5,4.4,4.3,4.3,4.3,5.4,5.8,6.2,5.4,..


In [214]:
wdi = {'GDP_pc': GDP_pc,
       'employment': employment,
       'lfp': lfp,
       'hci': hci,
       'undernourishment': undernourishment,
       'mortality': mortality,
       'primary_enroll': primary_enroll,
       'secondary_enroll': secondary_enroll,
       'tertiary_enroll':tertiary_enroll,
       'maternal_death_risk': maternal_death_risk,
       'mmr': mmr,
       'DPT_immunization': DPT_immunization,
      'population_growth': population_growth}

In [215]:
for series, data in wdi.items():
    data = data.rename(columns = {'Country Code': 'iso_o', 'Country Name':'country'}).iloc[:265, :]
    data = data.drop(columns= ['Series Name', 'Series Code'])
    data.columns = data.columns.str.replace(r' \[YR\d+\]', '', regex=True)
    data.iloc[:, 2:] = data.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').replace('..', np.nan).round(3)
    data_long = data.set_index(['iso_o', 'country']).stack().reset_index()
    data_long = data_long.rename(columns = {'level_2': 'year'})
    data_long['year'] = pd.to_numeric(data_long['year'], errors='coerce')
    data_long = data_long[(data_long['year'] >= 1990) & (data_long['year'] <= 2020)]
    wdi[series] = data_long
    globals()[series] = data_long

In [216]:
hci.head(2)

,iso_o,country,year,0
0,AFG,Afghanistan,2020,90.391
1,ALB,Albania,2010,153.581


In [217]:
GDP_pc = GDP_pc.rename(columns = {0: 'GDP_pc'})
employment = employment.rename(columns = {0: 'employment'})
lfp = lfp.rename(columns = {0: 'lfp'})
hci = hci.rename(columns = {0: 'hci'})
undernourishment = undernourishment.rename(columns = {0: 'undernourishment'})
mortality = mortality.rename(columns = {0: 'mortality'})
primary_enroll= primary_enroll.rename(columns = {0: 'primary_enroll'})
secondary_enroll = secondary_enroll.rename(columns = {0: 'secondary_enroll'})
tertiary_enroll = tertiary_enroll.rename(columns = {0: 'tertiary_enroll'})
maternal_death_risk = maternal_death_risk.rename(columns = {0: 'maternal_death_risk'})
mmr = mmr.rename(columns = {0: 'mmr'})
DPT_immunization = DPT_immunization.rename(columns = {0: 'DPT_immunization'})
population_growth = population_growth.rename(columns = {0: 'population_growth'})

In [218]:
merged_wdi = GDP_pc.merge(employment, on = ['iso_o', 'country', 'year'], how = 'left').merge(lfp, on = ['iso_o', 'country', 'year'], how = 'left').\
merge(hci, on = ['iso_o', 'country', 'year'], how = 'left').merge(undernourishment, on = ['iso_o', 'country', 'year'], how = 'left').\
merge(mortality, on = ['iso_o','country', 'year'], how = 'left').merge(primary_enroll, on = ['iso_o', 'country', 'year'], how = 'left').\
merge(secondary_enroll, on = ['iso_o', 'country', 'year'], how = 'left').merge(tertiary_enroll, on = ['iso_o', 'country', 'year'], how = 'left').\
merge(maternal_death_risk, on = ['iso_o', 'country', 'year'], how = 'left').merge(mmr, on = ['iso_o', 'country', 'year'], how = 'left').\
merge(DPT_immunization, on = ['iso_o', 'country', 'year'], how = 'left').merge(population_growth, on = ['iso_o', 'country', 'year'], how = 'left')

In [81]:
merged_wdi.isna().sum()

iso_o                     0
country                   0
year                      0
GDP_pc                    0
employment              900
lfp                     678
hci                    7328
undernourishment       3520
mortality               399
primary_enroll         3533
secondary_enroll       4466
tertiary_enroll        3169
maternal_death_risk     395
mmr                    6688
DPT_immunization        530
population_growth         1
nourishment            3520
infant_survival         399
maternal_survival      6688
dtype: int64

In [219]:
merged_wdi.head(2)

,iso_o,country,year,GDP_pc,employment,lfp,hci,undernourishment,mortality,primary_enroll,secondary_enroll,tertiary_enroll,maternal_death_risk,mmr,DPT_immunization,population_growth
0,AFG,Afghanistan,2000,308.318,42.885,37.344,NaN,NaN,110.1,NaN,NaN,NaN,10.166,1900.0,24.0,1.212
1,AFG,Afghanistan,2001,277.118,42.817,37.259,NaN,45.8,107.0,NaN,NaN,NaN,9.43,NaN,33.0,0.762


In [220]:
merged_wdi['nourishment'] = 100 - merged_wdi['undernourishment']
merged_wdi['infant_survival'] = 100 - (merged_wdi['mortality'] / 10)
merged_wdi['maternal_survival'] = 100 - (merged_wdi['mmr'] / 10)

In [221]:
print(f"Earliest year is {merged_wdi['year'].min()} and the latest year is {merged_wdi['year'].max()}.")

Earliest year is 1990 and the latest year is 2020.


In [224]:
merged_wdi[['hci', 'undernourishment', 'primary_enroll', 'secondary_enroll', 'tertiary_enroll']].notna().sum()

hci                  415
undernourishment    4223
primary_enroll      4210
secondary_enroll    3277
tertiary_enroll     4574
dtype: int64

#### From UNGI

In [82]:
idealpoints = pd.read_csv('../data/raw/IdealpointestimatesAll_Apr2020.csv', encoding='ISO-8859-1')

In [83]:
idealpoints['year'] = idealpoints['session'] + 1945

In [84]:
idealpoints = idealpoints[['IdealPoint', 'Countryname', 'year', 'iso3c', 'session']]

In [85]:
idealpoints = idealpoints.rename(columns = {'Countryname': 'country'})

In [86]:
idealpoints = idealpoints.rename(columns = {'IdealPoint': 'idealpoint'})
idealpoints = idealpoints.rename(columns = {'iso3c': 'iso_o'})

In [87]:
idealpoints['country'].nunique()

199

In [88]:
idealpoints['year'].unique()

array([1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956,
       1957, 1958, 1959, 1960, 1961, 1962, 1963, 1965, 1966, 1967, 1968,
       1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979,
       1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990,
       1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001,
       2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012,
       2013, 2014, 2015, 2016, 2017, 2018, 2019], dtype=int64)

In [89]:
idealpoints= idealpoints[idealpoints['year'] >=1990]

In [90]:
idealpoints.tail(2)

,idealpoint,country,year,iso_o,session
10467,0.085118,Samoa,2018,WSM,73
10468,0.031854,Samoa,2019,WSM,74


#### From UNDP

For UNDP datasets, the following steps are applied within a for loop:
1. The column name 'countryIsoCode' is changed to 'iso_o' 
2. Meta data columns are dropped. 
3. Years other than the range from 1990 to 2020 are trimmed out.

In [91]:
gii = pd.read_excel("../data/raw/hdr-data.xlsx")
lfp_f = pd.read_excel("../data/raw/hdr-data (1).xlsx")
parliament_seat_f = pd.read_excel("../data/raw/hdr-data (2).xlsx")
hdi = pd.read_excel("../data/raw/hdr-data (3).xlsx")
life_exp = pd.read_excel("../data/raw/hdr-data (4).xlsx")
mean_schooling = pd.read_excel("../data/raw/hdr-data (5).xlsx")
coef_human_inequality = pd.read_excel("../data/raw/hdr-data (6).xlsx")
inequality_edu = pd.read_excel("../data/raw/hdr-data (7).xlsx")
inequality_income = pd.read_excel("../data/raw/hdr-data (8).xlsx")
inequality_le = pd.read_excel("../data/raw/hdr-data (9).xlsx")

In [92]:
undp = {'gii': gii, 'lfp_f':lfp_f, 'parliament_seat_f':parliament_seat_f, 'hdi': hdi, 'life_exp':life_exp, 
        'mean_schooling':mean_schooling,'coef_human_inequality':coef_human_inequality, 'inequality_edu':inequality_edu, 
        'inequality_income':inequality_income, 'inequality_le':inequality_le}

In [93]:
for series, data in undp.items():
    data = data.rename(columns = {'countryIsoCode': 'iso_o'})
    data = data.drop(columns= ['indexCode', 'index', 'dimension', 'indicatorCode', 'indicator', 'yearStr', 'actualValue', 'note'])
    data = data[(data['year'] >= 1990) & (data['year'] <= 2020)]
    wdi[series] = data
    globals()[series] = data

In [94]:
gii = gii.rename(columns = {'value': 'gii'})
lfp_f = lfp_f.rename(columns = {'value': 'lfp_f'})
parliament_seat_f = parliament_seat_f.rename(columns = {'value': 'parliament_seat_f'})
hdi = hdi.rename(columns = {'value': 'hdi'})
life_exp = life_exp.rename(columns = {'value': 'life_exp'})
mean_schooling = mean_schooling.rename(columns = {'value': 'mean_schooling'})
coef_human_inequality = coef_human_inequality.rename(columns = {'value': 'coef_human_inequality'})
inequality_edu = inequality_edu.rename(columns = {'value': 'inequality_edu'})
inequality_income = inequality_income.rename(columns = {'value': 'inequality_income'})
inequality_le = inequality_le.rename(columns = {'value': 'inequality_le'})

In [95]:
inequality_le.head(2)

,iso_o,country,year,inequality_le
0,AFG,Afghanistan,2010,34.075
1,AFG,Afghanistan,2011,33.224


In [96]:
# merging the datasets
merged_undp = gii.merge(lfp_f, on = ['iso_o', 'country', 'year'], how='left').merge(parliament_seat_f, on = ['iso_o','country', 'year'], how='left').\
              merge(hdi, on = ['iso_o','country', 'year'], how='left').merge(life_exp, on = ['iso_o','country', 'year'], how='left').\
              merge(mean_schooling, on = ['iso_o','country', 'year'], how='left').merge(coef_human_inequality, on = ['iso_o','country', 'year'], how='left').\
              merge(inequality_edu, on = ['iso_o','country', 'year'], how='left').merge(inequality_income, on = ['iso_o','country', 'year'], how='left').\
              merge(inequality_le, on = ['iso_o','country', 'year'], how='left')

In [97]:
merged_undp.head(2)

,iso_o,country,year,gii,lfp_f,parliament_seat_f,hdi,life_exp,mean_schooling,coef_human_inequality,inequality_edu,inequality_income,inequality_le
0,AFG,Afghanistan,2008,0.690,43.79,25.872,0.446,59.708,1.743,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,2009,0.696,36.85,25.926,0.458,60.248,1.816,NaN,NaN,NaN,NaN


In [98]:
merged_undp.isna().sum()

iso_o                       0
country                     0
year                        0
gii                         0
lfp_f                       0
parliament_seat_f           0
hdi                        52
life_exp                    0
mean_schooling             18
coef_human_inequality    2582
inequality_edu           2461
inequality_income        2484
inequality_le            2321
dtype: int64

In [99]:
merged_undp['country'].nunique()

169

In [100]:
print(f"Earliest year is {merged_undp['year'].min()} and the latest year is {merged_undp['year'].max()}.")

Earliest year is 1990 and the latest year is 2020.


### Constructing final attributes dataset 

In [101]:
attributes = merged_wdi.merge(merged_owd, on = ['iso_o', 'country', 'year'], how='left').merge(merged_undp, on = ['iso_o', 'country', 'year'], how='left')

In [102]:
regional_entries = ['Central and Southern Asia (SDG)', 'East Asia and Pacific (WB)','Eastern and South-Eastern Asia (SDG)',
 'Europe and Central Asia (WB)','Europe and Northern America (SDG)','Latin America and Caribbean (WB)','Latin America and the Caribbean (SDG)',
 'Low-income countries','Lower-middle-income countries', 'Middle East and North Africa (WB)', 'Middle-income countries','North America (WB)',
 'North Korea','Northern Africa and Western Asia (SDG)','Oceania (excluding Australia and New Zealand) (SDG)','Saint Helena','South Asia (WB)',
 'Sub-Saharan Africa (SDG)','Sub-Saharan Africa (WB)', 'Upper-middle-income countries', 'World', 'Sub-Saharan Africa', 'Middle income','East Asia & Pacific (IDA & IBRD countries)',
'Upper middle income','Channel Islands','OECD members','Europe & Central Asia','European Union','Sub-Saharan Africa (IDA & IBRD countries)', 'IBRD only',
 'Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)','Europe & Central Asia (IDA & IBRD countries)','Sub-Saharan Africa (excluding high income)',
'Africa Eastern and Southern', 'High income','Europe & Central Asia (excluding high income)', 'Other small states','Pre-demographic dividend', 'South Asia',
'IDA total','IDA only','Caribbean small states', 'Small states','Least developed countries: UN classification','East Asia & Pacific','Early-demographic dividend',
 'Euro area','Central Europe and the Baltics','Middle East, North Africa, Afghanistan & Pakistan','Latin America & Caribbean','Early-demographic dividend',
 'Euro area','Least developed countries: UN classification','East Asia & Pacific', 'Heavily indebted poor countries (HIPC)','Post-demographic dividend',
 'Pacific island small states','IDA blend','Lower middle income','Middle East, North Africa, Afghanistan & Pakistan (excluding high income)',
 'Late-demographic dividend','Latin America & the Caribbean (IDA & IBRD countries)','Northern Mariana Islands', 'East Asia & Pacific (excluding high income)',
 'Fragile and conflict affected situations','Latin America & Caribbean (excluding high income)','IDA & IBRD total','Low & middle income','South Asia (IDA & IBRD)',
 'Low income','Low & middle income','South Asia (IDA & IBRD)','Africa Western and Central','Arab World','North America']

In [103]:
attributes = attributes[~attributes['country'].isin(regional_entries)]

In [104]:
attributes['country'].nunique()

211

In [105]:
attributes.columns

Index(['iso_o', 'country', 'year', 'GDP_pc', 'employment', 'lfp', 'hci',
       'undernourishment', 'mortality', 'primary_enroll', 'secondary_enroll',
       'tertiary_enroll', 'maternal_death_risk', 'mmr', 'DPT_immunization',
       'population_growth', 'nourishment', 'infant_survival',
       'maternal_survival', 'literacy_rate', 'gini', 'region',
       'electoral_dem_index', 'government_effec_index',
       'political_corruption_index', 'gii', 'lfp_f', 'parliament_seat_f',
       'hdi', 'life_exp', 'mean_schooling', 'coef_human_inequality',
       'inequality_edu', 'inequality_income', 'inequality_le'],
      dtype='object')

In [106]:
extra_in_idealpoints = list(set(idealpoints['country'].unique()) - set(attributes['country'].unique()))
extra_in_attributes = list(set(attributes['country'].unique()) - set(idealpoints['country'].unique()))

In [107]:
len(extra_in_idealpoints)

35

In [108]:
len(extra_in_attributes)

50

In [109]:
attributes[attributes['country'] == 'North Macedonia'].head(1)

,iso_o,country,year,GDP_pc,employment,lfp,hci,undernourishment,mortality,primary_enroll,...,gii,lfp_f,parliament_seat_f,hdi,life_exp,mean_schooling,coef_human_inequality,inequality_edu,inequality_income,inequality_le
4186,MKD,North Macedonia,1990,3534.323,NaN,43.577,NaN,NaN,33.3,NaN,...,0.432,45.69,3.333,0.644,71.339,4.603,NaN,NaN,NaN,NaN


In [110]:
idealpoints[idealpoints['country'] == 'Macedonia'].head(1)

,idealpoint,country,year,iso_o,session
3441,0.610558,Macedonia,1993,MKD,48


In [111]:
idealpoints.loc[idealpoints['country'] == 'Brunei', 'country'] = 'Brunei Darussalam'
idealpoints.loc[idealpoints['country'] == 'São Tomé & Príncipe', 'country'] = 'Sao Tome and Principe'
idealpoints.loc[idealpoints['country'] == 'Trinidad & Tobago', 'country'] = 'Trinidad and Tobago'

In [112]:
idealpoints.loc[idealpoints['country'] == 'Congo - Kinshasa', 'country'] = 'Congo, Dem. Rep.'
idealpoints.loc[idealpoints['country'] == 'Congo - Brazzaville', 'country'] = 'Congo, Rep.'
idealpoints.loc[idealpoints['country'] == 'Yemen', 'country'] = 'Yemen, Rep.'

In [113]:
idealpoints.loc[idealpoints['country'] == 'Swaziland', 'country'] = 'Eswatini'

In [114]:
idealpoints.loc[idealpoints['country'] == 'St. Kitts & Nevis', 'country'] = 'St. Kitts and Nevis'

In [115]:
idealpoints.loc[idealpoints['country'] == 'Macedonia', 'country'] = 'North Macedonia'

In [116]:
idealpoints.loc[idealpoints['country'] == 'St. Vincent & Grenadines', 'country'] = 'St. Vincent and the Grenadines'

In [117]:
idealpoints.loc[idealpoints['country'] == 'Cape Verde', 'country'] = 'Cabo Verde'
idealpoints.loc[idealpoints['country'] == 'Somalia', 'country'] = 'Somalia, Fed. Rep.'
idealpoints.loc[idealpoints['country'] == 'Gambia', 'country'] = 'Gambia, The'
idealpoints.loc[idealpoints['country'] == 'Micronesia (Federated States of)', 'country'] = 'Micronesia, Fed. Sts.'
idealpoints.loc[idealpoints['country'] == 'Antigua & Barbuda', 'country'] = 'Antigua and Barbuda'
idealpoints.loc[idealpoints['country'] == 'Bahamas', 'country'] = 'Bahamas, The'
idealpoints.loc[idealpoints['country'] == 'Côte d\x92Ivoire', 'country'] = "Cote d'Ivoire"
idealpoints.loc[idealpoints['country'] == 'Slovakia', 'country'] = 'Slovak Republic'
idealpoints.loc[idealpoints['country'] == 'Bosnia & Herzegovina', 'country'] = 'Bosnia and Herzegovina'
idealpoints.loc[idealpoints['country'] == 'South Korea', 'country'] = 'Korea, Rep.'

In [118]:
idealpoints.loc[idealpoints['country'] == 'Vietnam', 'country'] = 'Viet Nam'
idealpoints.loc[idealpoints['country'] == 'Egypt', 'country'] = "Egypt, Arab Rep."
idealpoints.loc[idealpoints['country'] == 'Iran', 'country'] = "Iran, Islamic Rep."
idealpoints.loc[idealpoints['country'] == 'Turkey', 'country'] = "Turkiye"
idealpoints.loc[idealpoints['country'] == 'Syria', 'country'] = "Syrian Arab Republic"
idealpoints.loc[idealpoints['country'] == 'Myanmar (Burma)', 'country'] = "Myanmar"
idealpoints.loc[idealpoints['country'] == 'Venezuela', 'country'] = "Venezuela, RB"
idealpoints.loc[idealpoints['country'] == 'Laos', 'country'] = "Lao PDR"
idealpoints.loc[idealpoints['country'] == 'Russia', 'country'] = "Russian Federation"
idealpoints.loc[idealpoints['country'] == 'Kyrgyzstan', 'country'] = 'Kyrgyz Republic'

In [119]:
print(attributes['country'].nunique())
print(attributes['iso_o'].nunique())
print(attributes.shape)
print(idealpoints.shape)
print(idealpoints.columns)

211
211
(6255, 35)
(5549, 5)
Index(['idealpoint', 'country', 'year', 'iso_o', 'session'], dtype='object')


In [120]:
attributes = attributes.merge(idealpoints, on = ['iso_o', 'country', 'year'], how='left')

In [121]:
# Removing leading and trailing spaces
attributes[['country', 'iso_o']] = attributes[['country', 'iso_o']].apply(lambda x: x.str.strip())

In [122]:
print(attributes.index.min(), attributes.index.max())
print(attributes.shape)
# Seems the attribute dataset is in good shape

0 6254
(6255, 37)


In [167]:
attributes.columns

Index(['iso_o', 'country', 'year', 'GDP_pc', 'employment', 'lfp', 'hci',
       'undernourishment', 'mortality', 'primary_enroll', 'secondary_enroll',
       'tertiary_enroll', 'maternal_death_risk', 'mmr', 'DPT_immunization',
       'population_growth', 'nourishment', 'infant_survival',
       'maternal_survival', 'literacy_rate', 'gini', 'region',
       'electoral_dem_index', 'government_effec_index',
       'political_corruption_index', 'gii', 'lfp_f', 'parliament_seat_f',
       'hdi', 'life_exp', 'mean_schooling', 'coef_human_inequality',
       'inequality_edu', 'inequality_income', 'inequality_le', 'idealpoint',
       'session'],
      dtype='object')

In [168]:
attributes.to_csv('../data/cleaned/attributes.csv', encoding = 'utf-8', index=False)

### Migration data

In [125]:
mig = pd.read_csv("../data/raw/migration_imputed_RIKS_dec2021.csv")

In [126]:
mig['year'].unique()

array([1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970,
       1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981,
       1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992,
       1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003,
       2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014,
       2015, 2016, 2017, 2018, 2019, 2020], dtype=int64)

In [127]:
# Droping the metadata columns and out of study years
mig = mig.drop(columns = ['inflow','outflow'])
mig = mig[(mig['year'] >= 1990) & (mig['year'] <= 2020)]

In [128]:
print(mig['destination'].nunique())
print(mig['origin'].nunique())
print(mig.groupby(['origin', 'destination']).ngroups)
print(mig['year'].nunique())

237
240
54841
31


In [129]:
print(set(mig['origin'].unique()) - set(mig['destination'].unique()))
print(set(mig['destination'].unique()) - set(mig['origin'].unique()))

{'Pitcairn Islands', 'Holy See', 'Sahrawi Arab Dem. Rep.'}
set()


In [130]:
# The following three countries are in origin, but nobody ever migrated to these countries. Also, these are not in the attributes. 
extra = ['Holy See', 'Pitcairn Islands', 'Sahrawi Arab Dem. Rep.']

In [131]:
mig = mig[~mig['origin'].isin(extra)]

In [132]:
print(mig['destination'].nunique())
print(mig['origin'].nunique())
print(mig.groupby(['origin', 'destination']).ngroups)
print(mig['year'].nunique())

237
237
54790
31


In [133]:
extra_in_migration = list(set(mig['destination'].unique()) - set(attributes['country'].unique()))
extra_in_attributes = list(set(attributes['country'].unique()) - set(mig['destination'].unique()))

In [134]:
extra_in_attributes

['Viet Nam',
 'Venezuela, RB',
 'Congo, Rep.',
 'Eswatini',
 'Dominican Republic',
 'Congo, Dem. Rep.',
 'Somalia, Fed. Rep.',
 'United States',
 'Yemen, Rep.',
 'Isle of Man',
 'Egypt, Arab Rep.',
 'Macao SAR, China',
 'Slovak Republic',
 'Turkiye',
 'Kyrgyz Republic',
 'Lao PDR',
 'Syrian Arab Republic',
 'Puerto Rico (US)',
 'Hong Kong SAR, China',
 'Netherlands',
 'Micronesia, Fed. Sts.',
 'Central African Republic',
 'Bahamas, The',
 'Czechia',
 'Gambia, The',
 'Korea, Rep.',
 'Iran, Islamic Rep.']

In [135]:
len(extra_in_attributes)

27

In [136]:
len(extra_in_migration)

53

In [137]:
extra_in_migration

['Hong Kong; SAR China',
 'Falkland Islands',
 'Puerto Rico',
 'Cook Islands',
 'Bahamas; The',
 'Korea; Rep.',
 'Czechoslovakia',
 'Macao SAR; China',
 'Yemen; Rep.',
 'Dominican Rep.',
 'Gambia; The',
 'Kyrgyz Rep.',
 'Somalia',
 "Korea; Dem. People's Rep.",
 'Wallis and Futuna',
 'Martinique',
 'St. Helena',
 'Guadeloupe',
 'Congo; Dem. Rep.',
 'Netherlands; The',
 'French Guiana',
 'Czech Rep.',
 'Niue',
 'United States of America',
 "Lao People's Dem. Rep.",
 'Swaziland',
 'Mayotte',
 'Turkey',
 'Central African Rep.',
 'USSR Soviet Union',
 'Anguilla',
 'Serbia-Montenegro',
 'Micronesia; Fed. Sts.',
 'Iran; Islamic Rep.',
 'Northern Mariana Islands',
 'Yemen Arab Rep.',
 'Vietnam',
 'Gibraltar',
 'Tokelau',
 'Reunion',
 'Norfolk Island',
 'Netherlands Antilles',
 'Slovak Rep.',
 'Congo; Rep.',
 'Bonaire; Sint Eustatius and Saba',
 'St. Pierre and Miquelon',
 'Yugoslavia',
 'Montserrat',
 'Syrian Arab Rep.',
 'Taiwan',
 'Egypt; Arab Rep.',
 'British Virgin Islands',
 'Venezuela']

In [138]:
#mig[mig['iso_or'] == "IMN"].head(1)

In [139]:
#attributes[attributes['country'] == 'Isle of Man'].head(2)

In [140]:
#Yemen Arab Rep. and Yemen; Rep. are different countires, but they merged in 1990. So, I'm including onlt Yemen,Rep. 
mig.loc[mig['origin'] == 'Yemen; Rep.', 'origin'] = 'Yemen, Rep.'
mig.loc[mig['destination'] == 'Yemen; Rep.', 'destination'] = 'Yemen, Rep.'

In [141]:
attributes.loc[attributes['country'] == 'Somalia, Fed. Rep.', 'country'] = 'Somalia'

In [142]:
attributes.loc[attributes['country'] == 'Turkiye', 'country'] = "Turkey"
attributes.loc[attributes['country'] == 'Lao PDR', 'country'] = "Lao People's Dem. Rep."
attributes.loc[attributes['country'] == 'Slovak Republic', 'country'] = "Slovak Rep."
attributes.loc[attributes['country'] == 'Viet Nam', 'country'] = "Vietnam"
attributes.loc[attributes['country'] == 'Dominican Republic', 'country'] = 'Dominican Rep.'

In [143]:
attributes.loc[attributes['country'] == 'Eswatini', 'country'] = 'Swaziland'
attributes.loc[attributes['country'] == 'Congo, Dem. Rep.', 'country'] = "Congo; Dem. Rep."

In [144]:
attributes.loc[attributes['country'] == 'Gambia, The', 'country'] = "Gambia; The"
attributes.loc[attributes['country'] == 'Macao SAR, China', 'country'] = 'Macao SAR; China'
attributes.loc[attributes['country'] == 'Hong Kong SAR, China', 'country'] = "Hong Kong; SAR China"

In [145]:
attributes.loc[attributes['country'] == 'Micronesia, Fed. Sts.', 'country'] = "Micronesia; Fed. Sts."
attributes.loc[attributes['country'] == 'Bahamas, The', 'country'] = "Bahamas; The"
attributes.loc[attributes['country'] == 'Czechia', 'country'] = 'Czech Rep.'
attributes.loc[attributes['country'] == 'Venezuela, RB', 'country'] = "Venezuela"
attributes.loc[attributes['country'] == 'Puerto Rico (US)', 'country'] = "Puerto Rico"
attributes.loc[attributes['country'] == 'Congo, Rep.', 'country'] =  'Congo; Rep.'
attributes.loc[attributes['country'] == 'Egypt, Arab Rep.', 'country'] = 'Egypt; Arab Rep.'
attributes.loc[attributes['country'] == 'Kyrgyz Republic', 'country'] = 'Kyrgyz Rep.'

In [146]:
attributes.loc[attributes['country'] == 'Iran, Islamic Rep.', 'country'] = 'Iran; Islamic Rep.'
attributes.loc[attributes['country'] == 'Korea, Rep.', 'country'] = "Korea; Rep."
attributes.loc[attributes['iso_o'] == 'KOR', 'iso_o'] = "ROK"
attributes.loc[attributes['country'] == 'United States', 'country'] = 'United States of America'

In [147]:
attributes.loc[attributes['country'] == 'Central African Republic', 'country'] = "Central African Rep."
mig.loc[mig['iso_or'] == 'CEN', 'iso_or'] = "CAF"
mig.loc[mig['iso_des'] == 'CEN', 'iso_des'] = "CAF"

In [148]:
attributes.loc[attributes['country'] == 'Netherlands', 'country'] = "Netherlands; The"

In [149]:
attributes.loc[attributes['country'] == 'Syrian Arab Republic', 'country'] = "Syrian Arab Rep."

In [150]:
common = set(attributes['country'].unique()) & set(mig['destination'].unique())

In [151]:
len(common)

210

Czechoslovakia and the Czech Republic are not the same countries. Yemen Arab Rep. is only present in the migration data for the year 1990 as it has unified with South Yemen and the complete Yemen is only considered year.  Netherlands; The and Netherlands are the same country. But Netherlands Antilles is not present in the attributes, hence removing from the study. 

As the ISO codes are different in migration data from Alpha3 codes, I am using country names to merge the datasets. Hence, I am renaming the countries in the attribute dataframe, following the entries in the migration dataframe.

After renaming these countries, there remains 1 more elements in the list `extra_in_attributes` which is 'Isle of Man', not present in the migration data. Also, 27 countries are still in `extra_in_migration` list, most of those are islands and parent nations those have separation within the time range of this study. Hence, not present in attributes. 
I am keeping the 210 countries present in both datasets.

In [152]:
attributes = attributes[attributes['country'].isin(common)]

In [153]:
mig = mig[mig['origin'].isin(common)]
mig = mig[mig['destination'].isin(common)]

In [154]:
print(attributes['country'].nunique())
print(mig['origin'].nunique())
print(mig['destination'].nunique())

210
210
210


In [155]:
mig.shape

(1212488, 7)

In [156]:
# Removing self loops
mig = mig[mig['origin'] != mig['destination']] #There is no self-loop
mig.shape

(1212488, 7)

In [157]:
# Checking for duplicates and removing if any
mig_duplicate = mig.duplicated(subset=['origin', 'destination', 'year'], keep = 'last')
mig_unique = mig[~mig_duplicate]
mig_unique.shape 

(1212488, 7)

In [158]:
print(mig_unique.groupby(['origin', 'destination']).ngroups)
# So, there are 43500 country pairs

43500


In [159]:
mig_unique = mig_unique.reset_index(drop = True)

In [160]:
mig_unique.head(2)

,iso_or,origin,iso_des,destination,year,stock,flow
0,AAB,Antigua and Barbuda,ABW,Aruba,1990,32,2.0
1,AAB,Antigua and Barbuda,ABW,Aruba,1991,34,2.0


In [161]:
assert set(mig_unique['origin'].unique()) == set(mig_unique['destination'].unique()) == set(attributes['country'].unique())

In [162]:
edgelist_flow = mig_unique.pivot_table(index = ['origin', 'destination'], columns = 'year', values = 'flow', fill_value = 0).reset_index()

In [163]:
edgelist_stock = mig_unique.pivot_table(index = ['origin', 'destination'], columns = 'year', values = 'stock', fill_value = 0).reset_index()

In [164]:
edgelist_flow.head(2)

year,origin,destination,1990,1991,1992,1993,1994,1995,1996,1997,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,Afghanistan,Albania,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Afghanistan,Algeria,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,...,0.0,-1.0,0.0,0.0,-1.0,0.0,-1.0,0.0,0.0,0.0


In [165]:
edgelist_stock.head(2)

year,origin,destination,1990,1991,1992,1993,1994,1995,1996,1997,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,Afghanistan,Albania,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Afghanistan,Algeria,10.0,10.0,10.0,9.0,9.0,9.0,9.0,9.0,...,3.0,2.0,2.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0


In [166]:
mig_unique.to_csv('../data/cleaned/bilateral_migration.csv', encoding = 'utf-8', index=False)
attributes.to_csv('../data/cleaned/attributes.csv', encoding = 'utf-8', index=False)
edgelist_flow.to_csv('../data/cleaned/edgelist_flow.csv', encoding = 'utf-8', index=False)
edgelist_stock.to_csv('../data/cleaned/edgelist_stock.csv', encoding = 'utf-8', index=False)

### Constructing bilateral panel data

Steps to construct the panel dataset:
1. Generating a list of countries and a list of years
2. Constructing a balanced panel by creating all possible country–pair combinations for each year in the study period
3. Merging the blank panel with bilateral migration data using origin–destination–year identifiers
4. Creating a partner-attributes dataset by duplicating the country attributes data, appending a '_d' suffix to all variables, and renaming the 'origin' identifier to 'destination'
5. Merging destination-country attributes into the panel dataset
6. Computing absolute differences between origin and destination country development indicators

In [169]:
# Making country list and year list
country_list = mig_unique['origin'].unique()
print(len(country_list))
years = list(range(1990, 2021))

210


In [170]:
# Construcing black matrix
blank = pd.MultiIndex.from_product([country_list, country_list, years], names=['origin', 'destination', 'year'])
blank_panel = blank.to_frame(index = False)
blank_panel = blank_panel[blank_panel['origin'] != blank_panel['destination']]
print(blank_panel.shape)

(1360590, 3)


In [171]:
(210*210 - 210)*31

1360590

In [172]:
blank_panel.head(1)

,origin,destination,year
31,Antigua and Barbuda,Aruba,1990


In [173]:
# Merging migration data with the blank matrix
migration = blank_panel.merge(mig_unique, on = ['origin', 'destination', 'year'], how = 'left')

In [174]:
migration.head(2)

,origin,destination,year,iso_or,iso_des,stock,flow
0,Antigua and Barbuda,Aruba,1990,AAB,ABW,32.0,2.0
1,Antigua and Barbuda,Aruba,1991,AAB,ABW,34.0,2.0


In [175]:
assert migration.shape[0] == blank_panel.shape[0] 

In [176]:
# Merging the attributes data with the bilateral migration data on origin and year columns
attributes = attributes.rename(columns = {'country': 'origin'})
panel = migration.merge(attributes, on = ['origin', 'year'], how = 'left')
panel = panel.drop(columns = ['iso_or', 'iso_des'])

In [177]:
assert migration.shape[0] == blank_panel.shape[0] == panel.shape[0]

In [178]:
# Constructing a dataframe for the attributes of destination countries
partner_attributes = attributes.copy()

In [179]:
len(list(partner_attributes.columns))

37

In [180]:
partner_attributes = partner_attributes.drop(columns = ['iso_o'])

In [182]:
partner_attributes.columns

Index(['origin', 'year', 'GDP_pc', 'employment', 'lfp', 'hci',
       'undernourishment', 'mortality', 'primary_enroll', 'secondary_enroll',
       'tertiary_enroll', 'maternal_death_risk', 'mmr', 'DPT_immunization',
       'population_growth', 'nourishment', 'infant_survival',
       'maternal_survival', 'literacy_rate', 'gini', 'region',
       'electoral_dem_index', 'government_effec_index',
       'political_corruption_index', 'gii', 'lfp_f', 'parliament_seat_f',
       'hdi', 'life_exp', 'mean_schooling', 'coef_human_inequality',
       'inequality_edu', 'inequality_income', 'inequality_le', 'idealpoint',
       'session'],
      dtype='object')

In [183]:
destination_attributes = {'origin': 'destination', 'GDP_pc': 'GDP_pc_d', 'mortality': 'mortality_d', 'infant_survival':'infant_survival_d', 'maternal_survival':'maternal_survival_d',
'primary_enroll': 'primary_enroll_d', 'secondary_enroll': 'secondary_enroll_d', 'tertiary_enroll': 'tertiary_enroll_d', 'maternal_death_risk':'maternal_death_risk_d' , 'mmr': 'mmr_d', 
'lfp':'lfp_d', 'hci':'hci_d','DPT_immunization':'DPT_immunization_d','population_growth':'population_growth_d','undernourishment': 'undernourishment_d','nourishment': 'nourishment_d',
'employment':'employment_d','literacy_rate': 'literacy_rate_d', 'gini': 'gini_d', 'region': 'region_d', 'electoral_dem_index': 'electoral_dem_index_d','government_effec_index': 'government_effec_index_d',
'political_corruption_index': 'political_corruption_index_d', 'gii': 'gii_d', 'lfp_f': 'lfp_f_d', 'parliament_seat_f': 'parliament_seat_f_d','hdi': 'hdi_d',
'life_exp': 'life_exp_d','mean_schooling': 'mean_schooling_d', 'coef_human_inequality': 'coef_human_inequality_d', 'inequality_edu': 'inequality_edu_d',
'inequality_income': 'inequality_income_d','inequality_le': 'inequality_le_d', 'idealpoint': 'idealpoint_d'}

In [184]:
partner_attributes = partner_attributes.rename(columns = destination_attributes)

In [185]:
# Merging the partner attributes data with the bilateral migration data on destination and year columns
panel2 = panel.merge(partner_attributes, on = ['destination', 'year'], how = 'left')

In [186]:
len(list(panel2.columns))

74

In [187]:
panel3 = panel2[['origin', 'destination', 'year', 'stock', 'flow', 'iso_o', 'region', 'region_d', 'GDP_pc', 'GDP_pc_d','mortality','mortality_d','primary_enroll',
        'secondary_enroll', 'secondary_enroll_d', 'tertiary_enroll', 'tertiary_enroll_d', 'maternal_death_risk', 'maternal_death_risk_d', 'mmr', 'mmr_d',      
        'primary_enroll_d','literacy_rate','literacy_rate_d', 'gini','gini_d','electoral_dem_index','electoral_dem_index_d','nourishment','nourishment_d',
        'government_effec_index','government_effec_index_d','political_corruption_index','political_corruption_index_d','gii','gii_d','lfp_f','lfp_f_d',
        'parliament_seat_f', 'parliament_seat_f_d','hdi','hdi_d','life_exp','life_exp_d','mean_schooling','mean_schooling_d','coef_human_inequality',
        'coef_human_inequality_d','inequality_edu', 'inequality_edu_d', 'inequality_income','inequality_income_d','inequality_le','inequality_le_d', 
        'idealpoint','idealpoint_d','DPT_immunization','DPT_immunization_d','population_growth','population_growth_d', 'session_x','infant_survival',
        'infant_survival_d', 'maternal_survival','maternal_survival_d','lfp','lfp_d','hci','hci_d', 'employment','employment_d', 'undernourishment','undernourishment_d']]

In [188]:
len(list(panel3.columns))

73

In [189]:
panel3.dtypes

origin                 object
destination            object
year                    int64
stock                 float64
flow                  float64
                       ...   
hci_d                  object
employment             object
employment_d           object
undernourishment       object
undernourishment_d     object
Length: 73, dtype: object

In [190]:
cols_to_convert = ['GDP_pc', 'GDP_pc_d', 'mortality', 'mortality_d', 'primary_enroll', 'primary_enroll_d', 'mmr', 'mmr_d','infant_survival_d', 'maternal_survival','maternal_survival_d','lfp','lfp_d','hci','hci_d', 
'employment','employment_d', 'undernourishment','undernourishment_d', 'DPT_immunization', 'DPT_immunization_d','secondary_enroll', 'secondary_enroll_d', 'tertiary_enroll', 'tertiary_enroll_d', 
'maternal_death_risk', 'maternal_death_risk_d', 'population_growth', 'population_growth_d','nourishment','nourishment_d']

panel3[cols_to_convert] = panel3[cols_to_convert].apply(pd.to_numeric, errors='coerce')

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_30028\715748279.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  panel3[cols_to_convert] = panel3[cols_to_convert].apply(pd.to_numeric, errors='coerce')


In [398]:
# # Constructing column for the absolutes difference between the origin and destination countries different development indicators
# panel2['diff_GDP_pc'] = (panel2['GDP_pc'] - panel2['GDP_pc_d']).abs()
# panel2['diff_mortality'] = (panel2['mortality'] - panel2['mortality_d']).abs()
# panel2['diff_primary_enroll'] = (panel2['primary_enroll'] - panel2['primary_enroll_d']).abs()
# panel2['diff_GDP_pc_gr'] = (panel2['GDP_pc_gr'] - panel2['GDP_pc_gr_d']).abs()
# panel2['diff_literacy_rate'] = (panel2['literacy_rate'] - panel2['literacy_rate_d']).abs()
# panel2['diff_gini'] = (panel2['gini'] - panel2['gini_d']).abs()
# panel2['diff_elec_democracy'] = (panel2['electoral_dem_index'] - panel2['electoral_dem_index_d']).abs()
# panel2['diff_govt_effect'] = (panel2['government_effec_index'] - panel2['government_effec_index_d']).abs()
# panel2['diff_political_corruption'] = (panel2['political_corruption_index'] - panel2['political_corruption_index_d']).abs()
# panel2['diff_gii'] = (panel2['gii'] - panel2['gii_d']).abs()
# panel2['diff_lfp_f'] = (panel2['lfp_f'] - panel2['lfp_f_d']).abs()
# panel2['diff_parliament_seat_f'] = (panel2['parliament_seat_f'] - panel2['parliament_seat_f_d']).abs()
# panel2['diff_hdi'] = (panel2['hdi'] - panel2['hdi_d']).abs()
# panel2['diff_life_exp'] = (panel2['life_exp'] - panel2['life_exp_d']).abs()
# panel2['diff_mean_schooling'] = (panel2['mean_schooling'] - panel2['mean_schooling_d']).abs()
# panel2['diff_coef_human_inequ'] = (panel2['coef_human_inequality'] - panel2['coef_human_inequality_d']).abs()
# panel2['diff_inequality_edu'] = (panel2['inequality_edu'] - panel2['inequality_edu_d']).abs()
# panel2['diff_inequality_income'] = (panel2['inequality_income'] - panel2['inequality_income_d']).abs()
# panel2['diff_inequality_le'] = (panel2['inequality_le'] - panel2['inequality_le_d']).abs()

In [191]:
print(panel3.groupby(['origin', 'destination']).ngroups)
print(panel3['year'].nunique())
print(panel3.shape)

43890
31
(1360590, 73)


In [192]:
print(panel3['origin'].nunique())

210


In [193]:
(210*210 - 210)*31

1360590

In [194]:
# constructing a dyad_id for all unique pair of countries
panel3['dyad_id'] = panel3['origin'] + '-' + panel3['destination'] 

In [195]:
panel3.head(2)

,origin,destination,year,stock,flow,iso_o,region,region_d,GDP_pc,GDP_pc_d,...,maternal_survival_d,lfp,lfp_d,hci,hci_d,employment,employment_d,undernourishment,undernourishment_d,dyad_id
0,Antigua and Barbuda,Aruba,1990,32.0,2.0,ATG,NaN,NaN,13166.524,24763.654,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Antigua and Barbuda-Aruba
1,Antigua and Barbuda,Aruba,1991,34.0,2.0,ATG,NaN,NaN,13385.011,25460.363,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Antigua and Barbuda-Aruba


In [196]:
panel3.to_csv('../data/cleaned/panel_migration.csv', encoding = 'utf-8', index=False)